# Snowflake Managed MCP Server — Live Demo

A CX analyst, working in their own agent (Claude Code), investigates a live customer escalation: they search reviews, quantify the sales impact, and open a **refund-approval case** — which a **human** signs off on before any money moves.

**Why MCP and not a scheduled job?** A nightly "flag bad SKUs" rule *should* be a Snowflake Task. MCP earns its place when a **human asks unpredictable questions in the agent they already live in**, and that agent composes your governed Snowflake tools on the fly. Schedule the predictable; expose over MCP the ad hoc, human-in-the-loop work.

## The objects behind this demo (already created)

| Object | What it is |
|---|---|
| `MCP_HOL.SUPPORT.REVIEWS` | customer reviews |
| `MCP_HOL.SALES.SALES_FACT` | orders / units / revenue / refunds |
| `MCP_HOL.SUPPORT.TICKETS` | refund-approval **cases** (PENDING_APPROVAL → APPROVED) |
| `MCP_HOL.SUPPORT.SEARCH_REVIEWS` | Cortex Search service over the reviews |
| `MCP_HOL.SALES.SALES_SV` | semantic view for Cortex Analyst |
| `MCP_HOL.SUPPORT.CREATE_TICKET` | procedure: opens a case (Zendesk-ready), **does not refund** |
| `MCP_HOL.SUPPORT.APPROVE_CASE` | procedure: the **human** approves → refund released |
| `MCP_HOL.AGENTS.MCP_HOL` | the MCP server — exposes search / analyst / open-case, **not** approve |

## Section 1 — The data

Raw reviews for the product under investigation, then a Cortex Search preview (testing-only — the agent hits this via the `search_reviews` tool).

In [ ]:
SELECT PRODUCT, RATING, REVIEW_TEXT, REVIEW_DATE
FROM MCP_HOL.SUPPORT.REVIEWS
WHERE PRODUCT = 'Summit Winter Jacket'
ORDER BY REVIEW_DATE DESC
LIMIT 15

In [ ]:
-- Testing-only preview. The agent calls this semantically via the search_reviews MCP tool.
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_REVIEWS',
  '{"query":"zipper keeps breaking","columns":["REVIEW_TEXT","PRODUCT","RATING"],"limit":5}'
) AS search_results

## Section 2 — Sales impact

Frame the exposure: refunds by product. The Summit Winter Jacket tops the list (~320 units, ~$9K refunds already).

In [ ]:
SELECT PRODUCT, SUM(UNITS) AS units, SUM(REFUND_AMT) AS refunds
FROM MCP_HOL.SALES.SALES_FACT
GROUP BY 1
ORDER BY refunds DESC

## Section 3 — The tools, exposed over MCP

One `CREATE MCP SERVER` statement turns three Snowflake objects into agent tools. Note what is **not** exposed: `APPROVE_CASE`. The agent can *open* a case; only a human can *approve* the refund.

```sql
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "search_reviews"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_REVIEWS"
  - name: "ask_sales_data"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "MCP_HOL.SALES.SALES_SV"
  - name: "create_support_ticket"   # opens a refund-approval case; does NOT refund
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CREATE_TICKET"
    config: { type: "procedure", warehouse: "COCO_WH", input_schema: { type: "object",
      properties: { order_id: {type: "string"}, issue: {type: "string"}, recommended_refund: {type: "number"} } } }
$$;
```

### Connect Claude Code (one-time)

```sql
ALTER USER ZHADA ADD PROGRAMMATIC ACCESS TOKEN mcp_hol_pat ROLE_RESTRICTION='ACCOUNTADMIN' DAYS_TO_EXPIRY=30;
```

```bash
claude mcp add --transport http snowflake-mcp \
  https://sfsenorthamerica-zhada-aws1.snowflakecomputing.com/api/v2/databases/MCP_HOL/schemas/AGENTS/mcp-servers/MCP_HOL \
  --header "Authorization: Bearer <PAT>"
```

For Claude Desktop / claude.ai, add the same URL as a Custom Connector with a Snowflake OAuth security integration.

## Section 4 — Baseline: open cases BEFORE the agent runs

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, EXTERNAL_CASE, APPROVED_BY
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Section 5 — RUN THE AGENT (live, in Claude Code)

Switch to **Claude Code** with `snowflake-mcp` connected and paste an ad-hoc request — the kind you'd never pre-script:

---

> *A customer emailed that the zipper on their Summit Winter Jacket (order ORD-10001) broke. Is this a known problem, and how big is it? If it's a real defect, open a refund-approval case for their order with a recommended refund.*

---

**Watch the agent compose your tools:**

1. `search_reviews` — finds the zipper-defect theme across 40+ reviews
2. `ask_sales_data` — quantifies it: ~320 units, ~$9K refunds already
3. `create_support_ticket` — opens a **PENDING_APPROVAL** case recommending the refund

The agent stops there. It has no tool to approve or move money — that boundary is enforced by what you exposed, not by a prompt.

## Section 6 — The case the agent opened (PENDING approval)

Re-run: a new case appears, `STATUS = PENDING_APPROVAL`, with the agent's recommended refund and (when reachable) the Zendesk case id.

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, EXTERNAL_CASE
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Section 7 — The human approves (this is the step the agent can't do)

A CX manager reviews the case and approves it. Approval is what releases the refund downstream — a privileged action deliberately kept out of the agent's tool surface. Replace the id with the case from Section 6.

In [ ]:
CALL MCP_HOL.SUPPORT.APPROVE_CASE(
  (SELECT TICKET_ID FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC LIMIT 1)
)

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, APPROVED_BY, APPROVED_AT
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Recap

- **Why MCP, not a cron job** — this was an *ad hoc*, human-driven investigation across reviews + sales + a case system, in the agent the analyst already works in. A schedule handles the predictable; MCP handles the unpredictable questions a person actually asks.
- **Build once, any client, your RBAC** — three Snowflake objects became tools with one `CREATE MCP SERVER`; every call runs under the caller's role.
- **The agent acted, but within a boundary** — it opened a governed case and recommended a refund; only a human could approve it. That boundary is the tool surface you chose to expose.

## Appendix A — The five MCP tool types

One MCP server can expose up to 50 tools. Every call runs under the **connecting user's DEFAULT_ROLE** (no secondary roles) on their DEFAULT_WAREHOUSE.

| Type | What it exposes | Used here |
|---|---|---|
| `CORTEX_SEARCH_SERVICE_QUERY` | Semantic search over a Cortex Search service | `search_reviews` |
| `CORTEX_ANALYST_MESSAGE` | Natural-language → SQL over a semantic view | `ask_sales_data` |
| `GENERIC` | Wrap one UDF/procedure as a typed action | `create_support_ticket` |
| `SYSTEM_EXECUTE_SQL` | Run SQL directly | — (see below) |
| `CORTEX_AGENT_RUN` | Expose an entire Cortex Agent as one tool | — (see Appendix C) |

**`SYSTEM_EXECUTE_SQL` config** — powerful, so scope it deliberately:
- `read_only`: `true` = SELECT only (default) · `false` = writes + DDL
- `query_timeout`: max seconds per query
- `warehouse`: compute to run on (else the user's default)

**`GENERIC` config** — the safest 'action' tool; one named operation, typed inputs, no freehand SQL:
- `identifier`: the UDF or procedure to wrap
- `type`: `function` (UDF) or `procedure`
- `input_schema`: the typed args the agent must fill
- `warehouse` / `query_timeout`: compute + limit

## Appendix B — How `create_support_ticket` reaches Zendesk (External Access)

The GENERIC tool wraps a Python procedure that makes a governed outbound call. The reach-out is built from a network rule + secret + external access integration — the agent never sees the credentials.

```sql
-- 1) allow egress to the CX host
CREATE NETWORK RULE ZENDESK_RULE MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('acme.zendesk.com');
-- 2) store the credential
CREATE SECRET ZENDESK_CRED TYPE=PASSWORD USERNAME='agent@acme.com/token' PASSWORD='<api_token>';
-- 3) bind them into an integration
CREATE EXTERNAL ACCESS INTEGRATION ZENDESK_MCP_HOL_INT
  ALLOWED_NETWORK_RULES=(ZENDESK_RULE) ALLOWED_AUTHENTICATION_SECRETS=(ZENDESK_CRED) ENABLED=TRUE;

-- 4) the procedure the tool wraps: opens a PENDING case, does NOT refund
CREATE PROCEDURE SUPPORT.CREATE_TICKET(order_id STRING, issue STRING, recommended_refund FLOAT)
  RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION=3.11 HANDLER='run'
  EXTERNAL_ACCESS_INTEGRATIONS=(ZENDESK_MCP_HOL_INT)
  PACKAGES=('snowflake-snowpark-python','requests') SECRETS=('cred'=ZENDESK_CRED)
AS $$
import _snowflake, requests
def run(session, order_id, issue, recommended_refund):
    c = _snowflake.get_username_password('cred')
    r = requests.post('https://acme.zendesk.com/api/v2/tickets.json',
        auth=(c.username, c.password),
        json={'ticket': {'subject': 'Refund approval: order ' + str(order_id),
                         'comment': {'body': issue + ' | $' + str(recommended_refund) + ' | PENDING_APPROVAL'}}})
    # ...insert a PENDING_APPROVAL row into SUPPORT.TICKETS with the Zendesk case id...
$$;
```

The matching `APPROVE_CASE(ticket_id)` procedure — the human step that releases the refund — is **not** exposed over MCP.

## Appendix C — Other ways to expose over MCP

You are not limited to hand-picked tools. Two shortcuts:

**Expose an entire Cortex Agent** (runs in Snowflake):
```sql
CREATE MCP SERVER support_api FROM SPECIFICATION $$
tools:
  - name: "support_agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "db.agents.support_agent"
$$;
```

**Expose Cortex Code (CoCo) itself** from your machine:
```bash
cortex mcp serve -c my_conn --bypass
```
...which hands the client `cortex_code_agent`, `cortex_analyst_query`, `cortex_search_objects`, plus `agents_list` / `search_docs`.